# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Appendix A: *Constructing a Mesh Network*
##### Version Number: 2.0
---
### Contents  
> 1. *ArcGIS workflow* 
> 2. *Visualization*
---
### Notes

- ArcGIS workflow for constructing a mesh network of sampling points equally spaced throughout California.
---
### Inputs
- `California shape file` - California boundaries
- `elevation raster` - Raster to calculate elevation for sampling points
- `CDFW regions` - California defined regions that offer different climate conditions
---
### Outputs  
- `sampling_points.csv` - completed mesh network 
---
### User Created Dependencies  

### ArcGIS Pro Workflow — Building the Mesh Network

*Projection Used: CRS: NAD 1983 California Statewide Lambert (meters)*

#### 1. Load sampling points, weather stations, and elevation raster
Inputs: `California shape file`, `elevation raster`, `CDFW regions`\
Tool: Add layers via Catalog → Folders/Databases → Add to Map\

#### 2. Create mesh network clipped to california
Input: `California shape file` layer\
Tool: Analysis → Tools → Create Fishnet\
Output: `Sampling_Points` layer
> Notes: 
> - cell height and width = 0.5
> - Creates equally spaced samples 46 Km apart throughout California

#### 3. Extract latitude and longitude into fields
Input: `Sampling_Points` layer\
Tool: Analysis → Tools → Calculate Geometry Attributes\
Output: `Sampling_Points` with Latitude and Longitude Fields\
> Notes: Projection used WGS 1984 Web Mercator (Auxiliary Sphere)

#### 4. Assign region ID to each sampling point
Input: `Sampling_points` and `CDFW_regions` layers\
Tool: Analysis → Tools → Overlay → Spatial Join\
Output: `Sampling_Points` with Region_ID and Region_Name fields
> Notes:
> - Target Features: sampling points
> - Join Features: CDFW regions

#### 5. Extract elevation values to points
Input: `Sampling_Points` and `Elevation` Raster\
Tool: Analysis → Tools → Spatial Analyst Tools → Extraction → Extract Values to Points\
Output: `Sampling_Points`
> Notes: Adds a new numeric field like Elevation to each point.

#### 6. Assign region to weather stations
Input: `Weather Stations` and `CDFW_Regions` Layers\
Tool: Analysis → Tools → Overlay → Spatial Join\
Output: `Weather_stations_with_Region`
> Notes:
> - Target Features: weather stations
> - Join Features: CDFW regions

#### 7. Find the closest Weather station for each sample
Input: `Sampling_Points` and `Weather_stations_with_Region`\
Tool: Analysis → Tools → Overlay → Spatial Join\
Output: `Sampling_points_with_Stations`
> Notes:
> - Input Features: sampling points with regions
> - Near Features: weather stations with regions
> - Match Option: Closest
> - Match Attributes: Region_ID
> - Attaches ID and Name of closest weather station in the same region.

#### 8. Create buffer around points
Input: `Sampling_points_with_Stations`\
Tool: Analysis → Tools → Buffer\
Output: `Sampling_Points_Buffered`
> Notes:
> - Buffer size is 36 Km
> - This ensures complete coverage of California but does result in minor overlap

#### 9. Calculate area of WUI layer
Input: `WUI`\
Tool: Calculate Geometry Atrributes\
Output: `WUI`
> Notes: 
> - Adds area field in square meters

#### 10. Intersect WUI layer with buffer layer
Input: `Sampling_Points_Buffered`, `WUI`\
Tool: Analysis → Tools → Intersect\
Output: `Sampling_Points_with_WUI`
> Notes: 

#### 11. Calculate WUI area within buffered radius of sampling points
Input: `Sampling_Points_with_WUI`\
Tool: Analysis → Tools → Summarize Within\
Output: `Sampling_Points_with_WUI`
> Notes: Perform three times once for each WUI zone
> - Interface, Intermix, Influence
> - Adds three new fields containing the total area each zone in a buffer

#### 12. Attach summed areas to sampling points
Input: `Sampling_Points_with_WUI`, `Sampling_points_with_Stations`\
Tools: Analysis → Tools → Spatial Join\
Output: `Final_Sampling_Points`
> Notes:
> - Target Features: your sampling points
> - Join Features: your buffer polygons with SUM_Clipped_Area
> - Join Operation: JOIN_ONE_TO_ONE (since one point is in one buffer)
> - Match Option: INTERSECT
> - Joins the three summed area fields with the point in the center of each buffer

#### 13. Export the final mesh layer
Tool: Data → Export to Table

<img src="../data/maps/mesh.jpg" width="800">

#### Final Product

- A complete mesh layer that combines:
- Sampling points
- Raster elevation values
- Nearest weather station information
- Region attributes
- *Interface*, *Intermix*, *Influence* zones areas summed up in a 36Km Buffer Radius 

- sampling points spaced 46 km apart

USDA CALVEG Zones - Ecoregions
https://data.fs.usda.gov/geodata/edw/datasets.php?dsetCategory=biota

### 1.3 Fire Station Metadata - `i04_CIMIS_Weather_Stations.csv`

Contains geographic and descriptive information for all relevant CIMIS weather stations in California. The dataset includes spatial data. including `latitude`, `longitude`, `station ID`, `elevation`, and `regional information`. These are merged with the weather readings to analyze regional weather trends and mapping results.

Source: [California Department of Water Resources – CIMIS](https://cimis.water.ca.gov/)  
Accessed and processed in accordance with public data use policies.

**initial observations:**
- `OBJECTID` represents station ID to match with CIMIS data
- `County`, `Latitude`, `Longitude`, `Elevation` only variables needed from this dataset and all appear to have zero null values.
- `County` field is different case than damages dataset.